# Coffee17 B0/C2 Internal OOF V1
Inference only. Attach original Coffee17 dataset and saved recovery notebook output containing best.pt and last.pt. This is retrospective internal OOF, not independent confirmation. No new training or checkpoint deletion.

In [ ]:
import os, sys, subprocess, tempfile, json, zipfile, hashlib
from pathlib import Path
import torch
PIN="cdef70e2d56eabbe1c83dbdaf2e000b65375176d"
INPUT=Path("/kaggle/input")
WORK=Path("/kaggle/working")
assert INPUT.is_dir() and WORK.is_dir()
if not torch.cuda.is_available():
    raise RuntimeError("Enable Kaggle GPU")
print("TRAINING: DISABLED")
print("EVALUATION: INTERNAL OOF ONLY")
print("CHECKPOINT DELETION: DISABLED")
print("PIN:",PIN)
REPO=Path(tempfile.mkdtemp(prefix="coffee17-internal-oof-src-"))/"repo"
subprocess.run(["git","clone","--quiet","--no-checkout","https://github.com/ediprin/coffee-bean-classification.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--quiet","--detach",PIN],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(REPO/"requirements/preprocessing-study.txt")],check=True)
ENV=dict(os.environ)
ENV["PYTHONPATH"]=str(REPO/"src")+os.pathsep+ENV.get("PYTHONPATH","")
sys.path.insert(0,str(REPO/"src"))
RUNNER=REPO/"src/bilinear_lmmd/experiments/run_c2_internal_oof_v1.py"
compile(RUNNER.read_text(),str(RUNNER),"exec")
from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.experiments.run_c2_internal_oof_v1 import AUTH_SHA
from bilinear_lmmd.core.reproducibility import sha256_file

# Select exactly the audited saved recovery output, not a compact analysis ZIP.
candidates=sorted(p for p in INPUT.rglob("recovery_authority.json") if p.is_file() and "datasets" not in p.relative_to(INPUT).parts)
matches=[p.parent for p in candidates if sha256_file(p)==AUTH_SHA]
if len(matches)!=1:
    print("RECOVERY AUTHORITIES FOUND:",[str(p) for p in candidates])
    raise RuntimeError("Attach exactly one saved B0/C2 recovery notebook output containing all checkpoint files.")
RECOVERY=matches[0]
print("RECOVERY SAVED OUTPUT:",RECOVERY)
# Same dataset-tree selector as recovery revision V2.
SLUG="coffee-green-bean-with-17-defects-original"
DATASETS_INPUT=INPUT/"datasets"
preferred=[DATASETS_INPUT/"sujitraarw"/SLUG,INPUT/SLUG]
present=[p for p in preferred if p.is_dir()]
if len(present)>1:
    raise RuntimeError(f"Multiple original dataset mounts: {present}")
if present:
    DATASET=present[0]
    samples=discover_directory_samples(DATASET)
else:
    valid=[]
    if DATASETS_INPUT.is_dir():
        for root in [DATASETS_INPUT]+sorted(p for p in DATASETS_INPUT.rglob("*") if p.is_dir()):
            try: samples=discover_directory_samples(root)
            except (ValueError,FileNotFoundError): continue
            if len(samples)==17 and sum(map(len,samples.values()))==979: valid.append((root,samples))
    deepest=[(root,samples) for root,samples in valid if not any(other!=root and root in other.parents for other,_ in valid)]
    if len(deepest)!=1:
        print("KAGGLE INPUT MOUNTS:",[str(p) for p in sorted(INPUT.iterdir())])
        raise RuntimeError("Attach one original Coffee17 dataset (not notebook outputs). Valid roots: "+str([str(x[0]) for x in deepest]))
    DATASET,samples=deepest[0]
assert len(samples)==17 and sum(map(len,samples.values()))==979
print("DATASET:",DATASET,"979/17")
OUT=WORK/"coffee17-b0-c2-internal-oof-v1"


In [ ]:
# Evaluation only: no optimizer, no training, no TTA.
# Existing output is preserved; successful evaluation can be packaged again.
SUMMARY=OUT/"analysis/internal_oof_v1.json"
if OUT.exists():
    if not SUMMARY.is_file():
        raise RuntimeError(f"Partial output preserved at {OUT}; do not discard it. Inspect execution log.")
    print("Existing complete OOF preserved. Skipping inference.")
else:
    subprocess.run([sys.executable,"-u","-m","bilinear_lmmd.experiments.run_c2_internal_oof_v1",
                    "--recovery",str(RECOVERY),"--dataset",str(DATASET),"--output",str(OUT),
                    "--required-commit",PIN,"--authorize-internal-oof"],cwd=REPO,env=ENV,check=True)
summary=json.loads(SUMMARY.read_text())
assert summary["protocol"]=="coffee17-b0-c2-internal-oof-v1"
assert summary["training_performed"] is False and summary["independent_confirmation"] is False
assert summary["n_unique_per_arm"]==965
print("DECISION:",summary["decision"])
print(json.dumps(summary["pooled_delta"],indent=2))


In [ ]:
# Package results separately, never copy/delete weights.
ZIP=WORK/"coffee17-b0-c2-internal-oof-v1.zip"
files=sorted(p for p in OUT.rglob("*") if p.is_file() and p.suffix in {".json",".csv"})
if ZIP.exists():
    with zipfile.ZipFile(ZIP) as z:
        assert z.testzip() is None
        assert set(z.namelist())=={str(p.relative_to(OUT)) for p in files}
        for p in files: assert hashlib.sha256(z.read(str(p.relative_to(OUT)))).hexdigest()==sha256_file(p)
else:
    tmp=ZIP.with_suffix(".tmp")
    with zipfile.ZipFile(tmp,"w",compression=zipfile.ZIP_DEFLATED) as z:
        for p in files: z.write(p,str(p.relative_to(OUT)))
    with zipfile.ZipFile(tmp) as z: assert z.testzip() is None
    tmp.replace(ZIP)
print("UPLOAD:",ZIP)
print("RECOVERY CHECKPOINTS UNCHANGED:",RECOVERY)
